## tl;dr

Use the bridge strategy: freeze the current baseline, introduce a no-behavior-change Prefill/Decode phase seam, then resume the original large-Prefill and measured-dispatch work. Do not continue low-yield Decode micro-tuning as the main line, and do not start a full multi-stream or triple-buffer rewrite.

## Context & Methods

The notebook compares the latest committed end-to-end diagnostic, the latest Nsight profile, the same-binary C16/C8 experiment, and the explicit unfinished roadmap items.

### Key Assumptions

- Unlocked-clock and single-profile measurements are diagnostic, not release claims.
- The wall-minus-kernel difference is an upper envelope, not a CUDA Graph forecast.
- C32/C64 benefits remain hypotheses until measured on longer prompt matrices.

## Data

Inputs are committed benchmark JSON files plus the bounded evidence extract next to this notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd

working = Path.cwd().resolve()
repo = next((candidate for candidate in (working, *working.parents) if (candidate / 'docs' / 'ROADMAP.md').exists()), None)
assert repo is not None, 'Repository root not found from the notebook working directory'
report_dir = repo / 'docs' / 'analysis' / 'priority-assessment-2026-07-23'
latest_path = repo / 'docs' / 'metadata' / 'qwen36-27b-gdn-rmsnorm-silu-gate-warp-tail-benchmark.json'
c16_path = repo / 'docs' / 'metadata' / 'qwen36-27b-c16-tensor-core-prefill-benchmark.json'
latest = json.loads(latest_path.read_text())
c16 = json.loads(c16_path.read_text())
evidence = pd.read_csv(report_dir / 'evidence.csv')
hotspots = pd.read_csv(report_dir / 'hotspots.csv')
evidence

,metric,value,unit,comparison_value,comparison_label,interpretation,source
0,Current TTFT,553.801000,ms,NaN,NaN,Latest max-26 B-C-C-B candidate average of pro...,latest-e2e
1,Current subsequent-token latency,111.434000,ms,NaN,NaN,Equivalent to 8.974 token/s for batch-one stea...,latest-e2e
2,Current decode throughput,8.973922,token/s,NaN,NaN,Derived as 1000 divided by 111.434 ms,latest-e2e
3,Current max-26 total generation,3339.769500,ms,NaN,NaN,Nineteen prompt tokens and twenty-six generate...,latest-e2e
4,Latest micro-optimization end-to-end delta,0.646000,ms,3339.1235,baseline total,A positive delta is a regression and the sourc...,latest-e2e
5,Latest profiled CUDA kernel time,3338.262016,ms,3367.7210,profiled generation wall time,The 29.459 ms difference is only an upper enve...,latest-profile
6,Wall-minus-kernel envelope,0.874745,percent,NaN,NaN,Not all of this gap is CUDA launch overhead,latest-profile
7,Top six M1 kernel groups,83.939275,percent of CUDA kernel time,NaN,NaN,Shows that current decode remains concentrated...,latest-profile
8,Prefill projection work,395.724576,ms,577.2520,profile TTFT,C16 plus M8 fallback plus C2 projection kernel...,latest-profile
9,C2 tail projection work,130.112416,ms,577.2520,profile TTFT,The tail alone accounts for 22.54 percent of t...,latest-profile


## Results

Recompute the highest-impact decision metrics from the committed records and reconcile them with the bounded extract.

In [2]:
averages = latest['detached_base_end_to_end_benchmark']['averaged_process_medians']
profile = latest['nsight_systems_max26_profile_comparison']
c16_delta = c16['same_binary_end_to_end_benchmark']['c16_versus_c8']

calculated = {
    'current_ttft_ms': averages['time_to_first_token']['candidate_milliseconds'],
    'current_subsequent_ms': averages['subsequent_token']['candidate_milliseconds'],
    'current_decode_tps': 1000.0 / averages['subsequent_token']['candidate_milliseconds'],
    'current_total_ms': averages['total']['candidate_milliseconds'],
    'latest_micro_e2e_delta_ms': averages['total']['candidate_milliseconds'] - averages['total']['baseline_milliseconds'],
    'wall_kernel_gap_ms': profile['profile_generation']['candidate_milliseconds'] - profile['all_cuda_kernels']['candidate_milliseconds'],
    'wall_kernel_gap_percent': 100.0 * (profile['profile_generation']['candidate_milliseconds'] - profile['all_cuda_kernels']['candidate_milliseconds']) / profile['profile_generation']['candidate_milliseconds'],
    'top_six_m1_share_percent': 100.0 * hotspots['total_ms'].sum() / profile['all_cuda_kernels']['candidate_milliseconds'],
    'c16_vs_c8_ttft_reduction_percent': -c16_delta['time_to_first_token_change_percent'],
}
pd.Series(calculated, name='value').to_frame()

,value
current_ttft_ms,553.801000
current_subsequent_ms,111.434000
current_decode_tps,8.973922
current_total_ms,3339.769500
latest_micro_e2e_delta_ms,0.646000
wall_kernel_gap_ms,29.458984
wall_kernel_gap_percent,0.874745
top_six_m1_share_percent,83.939275
c16_vs_c8_ttft_reduction_percent,25.468000


In [3]:
checks = {
    'TTFT reconciles': abs(calculated['current_ttft_ms'] - 553.801) < 1e-9,
    'Decode throughput reconciles': abs(calculated['current_decode_tps'] - 8.9739217833) < 1e-9,
    'Wall/kernel envelope reconciles': abs(calculated['wall_kernel_gap_percent'] - 0.8747453842) < 1e-9,
    'C16/C8 result reconciles': abs(calculated['c16_vs_c8_ttft_reduction_percent'] - 25.468) < 1e-9,
    'Top-six M1 share reconciles': abs(calculated['top_six_m1_share_percent'] - evidence.loc[evidence['metric'] == 'Top six M1 kernel groups', 'value'].iloc[0]) < 5e-7,
}
assert all(checks.values())
pd.Series(checks, name='passed').to_frame()

,passed
TTFT reconciles,True
Decode throughput reconciles,True
Wall/kernel envelope reconciles,True
C16/C8 result reconciles,True
Top-six M1 share reconciles,True


## Takeaways

- The latest Decode micro-change has no attributable end-to-end gain.
- The original roadmap already names large-Prefill coverage and measured dispatch boundaries as the next Phase 3 work.
- A thin phase seam lowers future rework without combining ownership, stream, workspace, and kernel changes in one risky rewrite.
- Double buffering, CUDA Graph, and C64 remain profile-gated follow-ups rather than immediate milestones.